# 05. Classical Machine Learning Benchmarks

Unified training and evaluation pipeline for classical ML baselines:
- **XGBoost**: Gradient boosted decision trees with `scale_pos_weight` for extreme class imbalance.
- **Random Forest**: Bagged ensembles with balanced class weighting.
- **Support Vector Machines (SVM)**: Linear and RBF kernel classifiers.
- **K-Nearest Neighbors (KNN)**: Distance-based local sequence similarity baseline.

Includes **decision-threshold optimization** (F1/PR-AUC maximizing cutoffs) and standardized metric logging.

In [ ]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    average_precision_score, matthews_corrcoef, precision_recall_curve
)
import xgboost as xgb

warnings.filterwarnings('ignore')
print("✓ ML libraries imported successfully.")

In [ ]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# Feature encoding to use: "onehot", "blosum", "aapc", or "hybrid"
ENCODING_MTHD = "onehot"

# Models to train: any subset of ["xgboost", "random_forest", "svm", "knn"]
MODELS_TO_RUN = ["xgboost", "random_forest", "svm", "knn"]

# Data paths
TRAIN_CSV_PATH = f"../data_engineered/{ENCODING_MTHD}/train_with_features_{ENCODING_MTHD}_split.csv"
VAL_CSV_PATH = f"../data_engineered/{ENCODING_MTHD}/val_with_features_{ENCODING_MTHD}_split.csv"
OUTPUT_BASE_DIR = "../output"

# Model hyperparameters
XGB_PARAMS = {"n_estimators": 150, "max_depth": 6, "learning_rate": 0.1, "n_jobs": -1, "random_state": 42}
RF_PARAMS = {"n_estimators": 100, "max_depth": 15, "n_jobs": -1, "random_state": 42, "class_weight": "balanced"}
KNN_PARAMS = {"n_neighbors": 7, "weights": "distance", "n_jobs": -1}

print("Configuration:")
print(f"  Encoding:    {ENCODING_MTHD}")
print(f"  Models:      {MODELS_TO_RUN}")
print(f"  Train path:  {TRAIN_CSV_PATH}")
print(f"  Val path:    {VAL_CSV_PATH}")

In [ ]:
def load_and_preprocess_split(train_path, val_path):
    """Load train/val splits and format numeric matrices."""
    print(f"Loading data from:\n  {train_path}\n  {val_path}")
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    
    label_cols = ['S-glutathionylation', 'S-nitrosylation', 'S-palmitoylation']
    metadata_cols = ['ID', 'Sequence'] + label_cols
    
    feature_cols = [c for c in train_df.columns if c not in metadata_cols]
    
    # Coerce object/bool columns
    X_train = train_df[feature_cols].copy()
    X_val = val_df[feature_cols].copy()
    
    for col in feature_cols:
        if X_train[col].dtype == object:
            X_train[col] = pd.to_numeric(X_train[col], errors='coerce').fillna(0)
            X_val[col] = pd.to_numeric(X_val[col], errors='coerce').fillna(0)
        elif X_train[col].dtype == bool:
            X_train[col] = X_train[col].astype(int)
            X_val[col] = X_val[col].astype(int)
            
    y_train = train_df[label_cols]
    y_val = val_df[label_cols]
    
    print(f"✓ Data loaded: Train {X_train.shape}, Val {X_val.shape}, Features: {len(feature_cols)}")
    return X_train, y_train, X_val, y_val, label_cols, feature_cols

In [ ]:
def optimize_threshold(y_true, y_probs):
    """Find optimal decision threshold maximizing F1 score."""
    best_thresh = 0.5
    best_f1 = 0.0
    
    for t in np.arange(0.05, 0.95, 0.02):
        preds = (y_probs >= t).astype(int)
        score = f1_score(y_true, preds, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thresh = t
            
    final_preds = (y_probs >= best_thresh).astype(int)
    return {
        "threshold": round(float(best_thresh), 3),
        "f1": float(f1_score(y_true, final_preds, zero_division=0)),
        "precision": float(precision_score(y_true, final_preds, zero_division=0)),
        "recall": float(recall_score(y_true, final_preds, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, final_preds)),
        "pr_auc": float(average_precision_score(y_true, y_probs)),
        "roc_auc": float(roc_auc_score(y_true, y_probs)) if len(np.unique(y_true)) > 1 else 0.5
    }

In [ ]:
def train_model(model_name, X_tr, y_tr_label, X_v, y_v_label):
    """Instantiate and train a single model for a single label."""
    pos_count = int(y_tr_label.sum())
    neg_count = len(y_tr_label) - pos_count
    spw = neg_count / max(pos_count, 1)
    
    if model_name == "xgboost":
        clf = xgb.XGBClassifier(**XGB_PARAMS, scale_pos_weight=spw, eval_metric='aucpr')
        clf.fit(X_tr, y_tr_label)
        probs = clf.predict_proba(X_v)[:, 1]
    elif model_name == "random_forest":
        clf = RandomForestClassifier(**RF_PARAMS)
        clf.fit(X_tr, y_tr_label)
        probs = clf.predict_proba(X_v)[:, 1]
    elif model_name == "svm":
        base_svc = LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)
        clf = CalibratedClassifierCV(base_svc, cv=3)
        clf.fit(X_tr, y_tr_label)
        probs = clf.predict_proba(X_v)[:, 1]
    elif model_name == "knn":
        clf = KNeighborsClassifier(**KNN_PARAMS)
        clf.fit(X_tr, y_tr_label)
        probs = clf.predict_proba(X_v)[:, 1]
    else:
        raise ValueError(f"Unknown model: {model_name}")
        
    metrics = optimize_threshold(y_v_label, probs)
    return clf, probs, metrics

In [ ]:
# Run complete benchmarking loop
if os.path.exists(TRAIN_CSV_PATH) and os.path.exists(VAL_CSV_PATH):
    X_train, y_train, X_val, y_val, label_cols, feature_cols = load_and_preprocess_split(TRAIN_CSV_PATH, VAL_CSV_PATH)
    
    all_results = []
    
    for model_name in MODELS_TO_RUN:
        print(f"\n{'='*70}\nBENCHMARKING: {model_name.upper()} ({ENCODING_MTHD})\n{'='*70}")
        out_dir = os.path.join(OUTPUT_BASE_DIR, model_name, ENCODING_MTHD)
        os.makedirs(out_dir, exist_ok=True)
        
        trained_models = {}
        val_probs_df = pd.DataFrame()
        model_metrics = []
        
        for label in label_cols:
            print(f"  Training for {label}...")
            clf, probs, metrics = train_model(model_name, X_train, y_train[label], X_val, y_val[label])
            trained_models[label] = clf
            val_probs_df[f"{label}_prob"] = probs
            val_probs_df[f"{label}_true"] = y_val[label].values
            
            metrics["model"] = model_name
            metrics["encoding"] = ENCODING_MTHD
            metrics["label"] = label
            model_metrics.append(metrics)
            print(f"    Optimal Threshold: {metrics['threshold']} | F1: {metrics['f1']:.4f} | PR-AUC: {metrics['pr_auc']:.4f} | ROC-AUC: {metrics['roc_auc']:.4f}")
            
        # Save model and predictions
        with open(os.path.join(out_dir, f"{model_name}_{ENCODING_MTHD}_models.pkl"), 'wb') as f:
            pickle.dump(trained_models, f)
        val_probs_df.to_csv(os.path.join(out_dir, f"{model_name}_{ENCODING_MTHD}_val_predictions.csv"), index=False)
        
        df_m = pd.DataFrame(model_metrics)
        df_m.to_csv(os.path.join(out_dir, f"{model_name}_{ENCODING_MTHD}_results.csv"), index=False)
        all_results.extend(model_metrics)
        
    summary_df = pd.DataFrame(all_results)
    summary_path = os.path.join(OUTPUT_BASE_DIR, f"classical_comparison_{ENCODING_MTHD}.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"\n✓ Benchmark complete! Summary saved to: {summary_path}")
else:
    print(f"⚠️ Split files not found:\n  {TRAIN_CSV_PATH}\n  {VAL_CSV_PATH}")

In [ ]:
# Summary Visualization
if 'summary_df' in locals() and not summary_df.empty:
    macro_summary = summary_df.groupby(['model', 'encoding'])[['f1', 'pr_auc', 'roc_auc', 'mcc']].mean().reset_index()
    print("\n" + "="*70 + "\nMACRO AVERAGE BENCHMARK SUMMARY\n" + "="*70)
    print(macro_summary.to_string(index=False))
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.barplot(data=macro_summary, x='model', y='f1', palette='viridis')
    plt.title(f"Macro F1 Comparison ({ENCODING_MTHD})")
    plt.ylabel("Macro F1")
    
    plt.subplot(1, 2, 2)
    sns.barplot(data=macro_summary, x='model', y='pr_auc', palette='magma')
    plt.title(f"Macro PR-AUC Comparison ({ENCODING_MTHD})")
    plt.ylabel("Macro PR-AUC")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE_DIR, f"classical_benchmark_{ENCODING_MTHD}.png"), dpi=300)
    plt.show()